# 01 · Explore documents: parsing and chunking quality

Checks the output of the first two pipeline steps:

1. `python -m plant_rag.ingest` → `data/processed/pages.jsonl` (+ `ingest_log.csv`)
2. `python -m plant_rag.chunk`  → `data/processed/chunks.jsonl`

Run those two commands first. This notebook only **reads** their outputs (no API calls, safe to re-run).
Kernel: the `plant-rag` venv (`%USERPROFILE%\envs\plant-rag`).

In [ ]:
# --- Setup and settings -------------------------------------------------------
import json
import random
import re
from collections import Counter

import matplotlib.pyplot as plt
import pandas as pd
import pymupdf

from plant_rag.config import load_config

cfg = load_config()                       # configs/config.yaml
P = cfg["paths"]
SAMPLE_SEED = 0                           # change to see other random chunks
SAMPLES_PER_TYPE = 2
SPACING_DOC, SPACING_PAGE = "KOSHA-C-C-11-2026", 4   # page used for the spacing comparison

pages = [json.loads(line) for line in open(P["processed_dir"] / "pages.jsonl", encoding="utf-8")]
chunks = pd.read_json(P["processed_dir"] / "chunks.jsonl", lines=True)
log = pd.read_csv(P["processed_dir"] / "ingest_log.csv", encoding="utf-8-sig")
manifest = pd.read_csv(P["manifest"], encoding="utf-8")
pd.set_option("display.max_colwidth", 80)
print(f"{len(manifest)} documents in manifest | {len(pages)} pages kept | {len(chunks)} chunks")

## 1. Pages and text per document

- `pdf_pages` vs `kept_pages`: pages skipped for having (almost) no text, i.e. scanned pages or figures.
- `hangul_share`: share of Hangul characters; ~0 for English documents.
- `bad_chars`: replacement characters (`\ufffd`) or private-use glyphs left in the text (should be 0).
- `equations`: `[수식]` placeholders. Equations from the Korean word processor's equation font can't be
  decoded, so ingest replaces them (see `unreadable_glyphs` rows in the log).

In [ ]:
rows = []
for m in manifest.itertuples():
    doc_pages = [p for p in pages if p["doc_id"] == m.doc_id]
    text = "".join(r["text"] for p in doc_pages for r in p["lines"])
    with pymupdf.open(P["raw_dir"] / m.file_name) as pdf:
        n_pdf = pdf.page_count
    rows.append({
        "doc_id": m.doc_id, "lang": m.language, "pdf_pages": n_pdf, "kept_pages": len(doc_pages),
        "chars": len(text),
        "hangul_share": round(len(re.findall(r"[\uac00-\ud7a3]", text)) / max(len(text), 1), 2),
        "bad_chars": len(re.findall(r"[\ufffd\ue000-\uf8ff]", text)),
        "equations": text.count(cfg["ingest"]["unreadable_placeholder"]),
        "removed_lines": int((log.doc_id.eq(m.doc_id) & log.event.str.startswith("removed")).sum()),
    })
page_stats = pd.DataFrame(rows)
page_stats

In [ ]:
# Pages skipped or with undecodable glyphs
log[~log.event.str.startswith("removed")].sort_values(["event", "doc_id", "page"])

## 2. Korean spacing fix

KOSHA PDFs contain no space characters. `ingest.py` rebuilds spaces from the gaps between characters.
Compare with PyMuPDF's plain text and with Kiwi's automatic spacing (which splits compound nouns,
e.g. 안전밸브 → 안전 밸브, so it is only used to decide spaces at line wraps).

In [ ]:
from kiwipiepy import Kiwi

with pymupdf.open(P["raw_dir"] / manifest.set_index("doc_id").loc[SPACING_DOC, "file_name"]) as pdf:
    plain = pdf[SPACING_PAGE - 1].get_text()
ours = "\n".join(r["text"] for p in pages if p["doc_id"] == SPACING_DOC and p["page"] == SPACING_PAGE
                 for r in p["lines"])
plain_body = plain[plain.find("1."):][:250]
ours_body = ours[ours.find("1."):][:250]
print("PyMuPDF plain text:\n", plain_body, "\n")
print("Gap-based spacing (ingest.py):\n", ours_body, "\n")
print("Kiwi automatic spacing on the plain text:\n", Kiwi().space(plain_body.replace("\n", " ")))

## 3. Header/footer removal

What `ingest.py` removed, grouped by document and rule (digits shown as `#`).
Check that only running headers, footers and page numbers appear here, not body text.

In [ ]:
removed = log[log.event.str.startswith("removed")].copy()
removed["pattern"] = removed.text.astype(str).str.replace(r"\d+", "#", regex=True).str[:60]
(removed.groupby(["doc_id", "event", "pattern"]).size().rename("count").reset_index()
        .sort_values(["doc_id", "count"], ascending=[True, False]))

## 4. Chunk sizes

Chunks are clauses (articles, numbered sections, lettered paragraphs). Long clauses are split at
sub-items and packed up to `max_tokens`; anything still too long becomes overlapping windows
(`clause_id` ends with `(part k/n)`). Token counts use the reranker's tokenizer.

In [ ]:
chunks["windowed"] = chunks.clause_id.str.contains(r"\(part \d+/\d+\)")
size_stats = chunks.groupby(["doc_id", "doc_type"]).agg(
    chunks=("chunk_id", "size"), tokens_median=("n_tokens", "median"), tokens_max=("n_tokens", "max"),
    tokens_min=("n_tokens", "min"), windowed=("windowed", "sum")).reset_index()
print("max_tokens =", cfg["chunking"]["max_tokens"], "| over limit:", int((chunks.n_tokens > cfg["chunking"]["max_tokens"]).sum()))
size_stats

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.5))
for kind, grp in chunks.groupby("doc_type"):
    ax.hist(grp.n_tokens, bins=30, alpha=0.55, label=f"{kind} (n={len(grp)})")
ax.set_xlabel("tokens per chunk (reranker tokenizer)")
ax.set_ylabel("chunks")
ax.set_title("Chunk length distribution")
ax.legend()
plt.tight_layout()
plt.show()

## 5. Structure checks

- CFR: paragraphs (a)–(p) and appendices A–D should all be present, in order.
- Korean rule: article numbers should only increase (부칙 restarts at 제1조).
- KOSHA: list of top-level sections per guide (compare with each guide's table of contents).

In [ ]:
def first_clause(cid: str) -> str:
    return re.sub(r"\s*\(part \d+/\d+\)$", "", cid)

cfr = chunks[chunks.doc_type == "cfr"].clause_id.map(first_clause)
letters = sorted({m.group(1) for c in cfr if (m := re.match(r"^\(([a-z])\)", c))})
appx = sorted({m.group(1) for c in cfr if (m := re.match(r"^Appendix ([A-Z])", c))})
print("CFR paragraphs:", "".join(letters), "| missing:", sorted(set("abcdefghijklmnop") - set(letters)))
print("CFR appendices:", appx)

law = chunks[chunks.doc_type == "kr_law"]
arts = [(int(m.group(1)), int(m.group(2) or 0)) for c in law.clause_id
        if (m := re.match(r"^제(\d+)조(?:의(\d+))?", c))]
unique_arts = list(dict.fromkeys(arts))
print(f"Korean rule: {len(unique_arts)} articles, 제{unique_arts[0][0]}조 … 제{unique_arts[-1][0]}조,",
      "monotonic:", all(a < b for a, b in zip(unique_arts, unique_arts[1:])))
print("Addenda chunks:", law[law.clause_id.str.startswith("부칙")].clause_id.tolist())

for doc_id, grp in chunks[chunks.doc_type == "kosha"].groupby("doc_id"):
    tops = list(dict.fromkeys(first_clause(c).split(" ")[0] for c in grp.clause_id))
    print(f"{doc_id}: {' '.join(tops)}")

## 6. Read random chunks

Read a few chunks end to end: is the text complete, correctly spaced, in reading order, and is
`clause_id` / page range right? Change `SAMPLE_SEED` to see others.

In [ ]:
rng = random.Random(SAMPLE_SEED)
for kind, grp in chunks.groupby("doc_type"):
    for idx in rng.sample(list(grp.index), k=min(SAMPLES_PER_TYPE, len(grp))):
        r = chunks.loc[idx]
        print(f"### [{kind}] {r.chunk_id} | {r.clause_id} | p{r.page_start}-{r.page_end} | {r.n_tokens} tokens")
        if r.section:
            print("section:", r.section)
        print(r.text[:600] + (" …" if len(r.text) > 600 else ""), "\n")

## Findings (run of 2026-10-13, 10 documents)

- All 10 PDFs have a text layer; 3 near-empty pages (figures) were skipped (385 → 382 pages).
- KOSHA spacing is rebuilt exactly from character gaps; Kiwi alone would split compound nouns.
- Header/footer removal only touches running headers, footers and page numbers. Known small losses:
  repeated table headers (CFR Appendix A "Chemical name / CAS / TQ"), and KOSHA C-C-37 `<별지 N>` form labels.
- Equations in KOSHA C-C-11 and C-C-93 can't be decoded (HWP equation font): 46 `[수식]` placeholders.
- 1,156 chunks; none over 500 tokens. CFR (a)–(p) + Appendices A–D found; Korean rule 제1조–제674조
  (687 numbered articles incl. 제N조의M) in order, plus 부칙.
- Known limitations: tables are flattened into text lines; KOSHA tables placed after the last section are
  attached to that section; line wraps in the Korean rule are joined with Kiwi, which sometimes adds a space
  inside a compound noun (압력용기 → 압력 용기). The OSHA 3132 booklet ends with ~45 chunks of contact directory and publication lists
  (pages 51–59) that are unlikely to answer safety questions.